# Week 5 – Vision (Doğukan Yeni): more Turkish plates + hard cases, retrain with augmentation

**Goal:** plate detector **mAP ≥ 0.75** and better recall on small / hard plates (plate_v1 recall was 0.73).

This notebook downloads the data **itself** from the internet:
| Source | What | How |
|---|---|---|
| Roboflow `guler-kandeger/plate-detection-vh2rk` | Turkish plate boxes (used by a public Turkish ANPR project) | Roboflow API |
| Roboflow `plakatanima-vnt3k/turkish-number-plates` | Turkish plate boxes, CC BY 4.0 | Roboflow API |
| Open Images V7 – *Vehicle registration plate* | Hard cases: far, angled, blurry, motorbikes, many countries | FiftyOne (no key) |
| Your plate_v1 dataset (optional) | So old data is not lost | Drive `ANPR/plate_dataset/data.yaml` |

You only need a **free Roboflow API key**: roboflow.com → Settings → *API Keys* → copy the Private API Key.

Runtime → Change runtime type → **T4 GPU**. Run cells one by one.

In [ ]:
!pip install -q ultralytics roboflow fiftyone

In [ ]:
import os
if not os.path.exists('anpr-project-en'):
    !git clone -q https://github.com/cemregonenc/anpr-project-en.git
else:
    !cd anpr-project-en && git pull -q
os.chdir('/content/anpr-project-en')
print(os.getcwd())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/ANPR'
PLATE_V1 = f'{D}/models/plate_best.pt'           # plate_v1 weights (for fine-tuning and comparison)
OLD_YAML = f'{D}/plate_dataset/data.yaml'        # plate_v1 dataset, optional
for p in [PLATE_V1, OLD_YAML]:
    print('OK     ' if os.path.exists(p) else 'MISSING', p)

## 1) Download Turkish plate datasets from Roboflow

In [ ]:
from getpass import getpass
from roboflow import Roboflow
rf = Roboflow(api_key=getpass('Roboflow API key: '))

ROBOFLOW_SETS = [('guler-kandeger', 'plate-detection-vh2rk'),
                 ('plakatanima-vnt3k', 'turkish-number-plates')]
downloaded = {}
for ws, proj in ROBOFLOW_SETS:
    try:
        p = rf.workspace(ws).project(proj)
        v = max(p.versions(), key=lambda v: int(v.version))       # latest version
        ds = v.download('yolov8', location=f'/content/datasets/{proj}')
        downloaded[proj] = f'{ds.location}/data.yaml'
        print('OK', proj, 'version', v.version)
    except Exception as e:
        print('FAILED', proj, '->', e)
downloaded

## 2) Download hard cases from Open Images (no key needed)
~600 images for train/valid and 200 separate **hard test** images.

In [ ]:
import fiftyone as fo, fiftyone.zoo as foz
CLS = ['Vehicle registration plate']
OI = '/content/datasets/openimages_plates'
OI_TEST = '/content/datasets/openimages_hard_test'

trainval = foz.load_zoo_dataset('open-images-v7', split='validation', label_types=['detections'],
                                classes=CLS, max_samples=600, shuffle=True, seed=51, dataset_name='oi_tv')
tr = trainval.take(480, seed=1)
va = trainval.exclude(tr.values('id'))
for view, split in [(tr, 'train'), (va, 'val')]:
    view.export(export_dir=OI, dataset_type=fo.types.YOLOv5Dataset, label_field='ground_truth', classes=CLS, split=split)

hard = foz.load_zoo_dataset('open-images-v7', split='test', label_types=['detections'],
                            classes=CLS, max_samples=200, shuffle=True, seed=7, dataset_name='oi_test')
hard.export(export_dir=OI_TEST, dataset_type=fo.types.YOLOv5Dataset, label_field='ground_truth', classes=CLS, split='test')
print(os.listdir(OI), os.listdir(OI_TEST))

## 3) Merge everything into one plate-only dataset
Only plate boxes are kept (class 0 = `plate`), duplicates are dropped. The table goes into `docs/experiments.md`.

In [ ]:
sources = [f'tr_{i}={y}' for i, y in enumerate(downloaded.values())] + [f'openimages={OI}/dataset.yaml']
if os.path.exists(OLD_YAML):
    sources.insert(0, f'old={OLD_YAML}')
args = ' '.join(f'--source "{s}"' for s in sources)
!python models/merge_datasets.py --out /content/datasets/plates_v2 $args

# hard test set as its own dataset (only for comparison)
!python models/merge_datasets.py --out /content/datasets/hard_test --source "hard={OI_TEST}/dataset.yaml"
MERGED = '/content/datasets/plates_v2/data.yaml'
HARD = '/content/datasets/hard_test/data.yaml'

## 4) Train plate_v2 with augmentation
Augmentation = new versions of each image during training: brightness/colour (`hsv_*`), rotation (`degrees`), zoom (`scale`), small perspective changes.
`fliplr=0` because a mirrored plate is not a real plate. Starts from plate_v1 if it is in Drive (faster), otherwise from `yolov8n.pt`.
About 1 hour on a T4.

In [ ]:
from ultralytics import YOLO
start = PLATE_V1 if os.path.exists(PLATE_V1) else 'yolov8n.pt'
print('start from', start)
model = YOLO(start)
model.train(data=MERGED, epochs=50, imgsz=640, batch=16,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,      # colour + brightness (night / sun)
            degrees=10, translate=0.1, scale=0.6,   # angle + distance
            shear=2.0, perspective=0.0005,
            fliplr=0.0, mosaic=1.0,
            project=f'{D}/runs', name='plate_v2', exist_ok=True)
V2 = f'{D}/runs/plate_v2/weights/best.pt'
!cp "$V2" "$D/models/plate_v2.pt"
print('saved', f'{D}/models/plate_v2.pt')

## 5) Old vs new on the **same** test sets

In [ ]:
import pandas as pd
rows = []
models_to_test = {'plate_v1': PLATE_V1, 'plate_v2': f'{D}/models/plate_v2.pt'}
for mname, w in models_to_test.items():
    if not os.path.exists(w):
        continue
    m = YOLO(w)
    if len(m.names) != 1:
        print(f'note: {mname} has classes {m.names}; plate must be class 0 for a fair score')
    for tname, y in [('merged test', MERGED), ('hard test (Open Images)', HARD)]:
        r = m.val(data=y, split='test', imgsz=640, verbose=False, plots=False)
        rows.append({'Model': mname, 'Test set': tname, 'mAP50': round(r.box.map50, 3),
                     'mAP50-95': round(r.box.map, 3), 'Precision': round(r.box.mp, 3), 'Recall': round(r.box.mr, 3)})
cmp = pd.DataFrame(rows)
cmp

In [ ]:
from datetime import date
print('Rows for docs/experiments.md:\n')
for _, r in cmp[cmp['Model'] == 'plate_v2'].iterrows():
    print(f"| {date.today()} | A | YOLOv8n (detect) | 50 | 640 | {r['mAP50']} | plate_v2 on {r['Test set']}. P={r['Precision']}, R={r['Recall']}, mAP50-95={r['mAP50-95']}. Turkish Roboflow sets + Open Images hard cases, augmentation on. |")
v2 = cmp[(cmp.Model == 'plate_v2') & (cmp['Test set'] == 'merged test')]
if len(v2):
    print('\nTarget mAP50 >= 0.75:', 'PASSED' if v2['mAP50'].iloc[0] >= 0.75 else 'NOT YET')
print('\nIf plate_v2 is better: copy ANPR/models/plate_v2.pt -> ANPR/models/plate_best.pt (keep plate_v1 as a backup).')

## 6) Look at mistakes on the hard test set (for the report)

In [ ]:
import glob, cv2, matplotlib.pyplot as plt
m = YOLO(f'{D}/models/plate_v2.pt')
imgs = sorted(glob.glob('/content/datasets/hard_test/test/images/*'))[:12]
res = m.predict(imgs, conf=0.25, verbose=False)
plt.figure(figsize=(16, 12))
for i, r in enumerate(res):
    plt.subplot(3, 4, i + 1); plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f'{len(r.boxes)} plates found', fontsize=9)
plt.tight_layout(); plt.show()